# Qwen-TTS 보이스 클로닝 실습 매뉴얼

부제: 구글 코랩에서 Qwen3-TTS Base 모델로 목소리를 복제해 보는 실습 교재

---

## 1. Qwen-TTS란?

Qwen-TTS는 Qwen 팀이 공개한 오픈소스 텍스트-투-스피치(TTS) 모델 시리즈다.
문장을 음성으로 바꾸는 기능뿐 아니라, 다음과 같은 기능을 지원한다.

- 다국어 음성 합성
- 자연어 지시를 통한 스타일 제어
- 스트리밍 기반 저지연 합성
- 참조 음성을 이용한 보이스 클로닝
- 미리 준비된 화자를 사용하는 TTS

공식 자료 기준으로 Qwen3-TTS는 다음 10개 언어를 지원한다.

Chinese, English, Japanese, Korean, German, French, Russian, Portuguese, Spanish, Italian

이 노트북은 그중 **보이스 클로닝**만 다룬다.

## 2. 이 노트북의 동작 방식

### 2.1 클로닝에 필요한 두 가지

Base 모델로 보이스 클로닝을 하려면 아래 두 입력이 필요하다.

- `ref_audio` — 복제할 목소리가 담긴 참조 음성 파일
- `ref_text` — 그 음성에 **실제로 들어 있는 문장**

둘이 어긋나면 품질이 눈에 띄게 떨어진다. 녹음한 그대로 받아써야 한다.

### 2.2 파일 이름 규칙

이 노트북은 **이름 하나만 입력하면** 참조 파일과 대본을 함께 찾아 읽는다.
그래서 두 파일의 이름(확장자 앞부분)이 서로 같아야 한다.

| 파일 | 역할 | 필수 |
|---|---|---|
| `애국가1절.m4a` | 참조 음성 → `ref_audio` | 필수 |
| `애국가1절.txt` | 그 음성의 대본 → `ref_text` | 선택 |

`애국가1절` 이라고만 입력하면 위 파일들을 자동으로 읽는다.
**`.txt`가 없으면 Whisper로 받아써서 자동으로 만들어 준다** (2.3 참고).
목소리를 추가하려면 같은 규칙으로 파일을 폴더에 넣기만 하면 된다.

### 2.3 대본 자동 생성 (STT)

영상이나 음성만 넣고 대본을 따로 만들지 않아도 된다.
`.txt`가 없으면 Whisper가 참조 음성을 받아쓰고, 결과를 `이름.txt`로 저장한다.

```
발표영상.mp4 만 넣음
        ↓
16kHz wav 변환 → Whisper 받아쓰기 → 발표영상.txt 저장 → ref_text로 사용
```

한 번 만들어진 `.txt`는 다음 실행부터 그대로 재사용하므로 STT를 다시 돌리지 않는다.

**받아쓴 결과는 반드시 확인할 것.** `ref_text`는 음성과 정확히 일치해야 하고,
STT가 틀리면 클로닝 품질이 그만큼 떨어진다. 화면에 출력된 대본이 녹음과 다르면
저장된 `.txt`를 직접 고친 뒤 6.2 셀을 다시 실행하면 된다.

관련 설정:

| 설정 | 의미 |
|---|---|
| `AUTO_STT` | `.txt`가 없을 때 자동 받아쓰기 여부. `False`면 대본이 없을 때 오류 |
| `STT_MODEL_ID` | 사용할 Whisper 모델. 느리면 `openai/whisper-small`로 낮춘다 |
| `STT_LANGUAGE` | 받아쓰기 언어 코드 (한국어는 `ko`) |
| `load_reference(이름, force_stt=True)` | 기존 `.txt`를 무시하고 다시 받아쓴다 |

### 2.4 지원 파일 형식

음성 파일과 **영상 파일** 모두 참조로 쓸 수 있다.
영상을 넣으면 ffmpeg가 오디오 트랙만 뽑아내므로 짝 규칙은 동일하다.

| 구분 | 확장자 |
|---|---|
| 음성 | `.m4a` `.mp3` `.wav` `.webm` `.ogg` `.flac` |
| 영상 | `.mp4` `.mov` `.mkv` `.avi` |

즉 `애국가1절.mp4` + `애국가1절.txt` 조합도 그대로 동작한다.

영상을 쓸 때 유의할 점:

- 카메라 마이크로 녹음된 경우가 많아 거리·잔향 때문에 품질이 떨어질 수 있다
- 여러 사람이 말하는 구간이 섞이면 클로닝 결과가 나빠진다
- 파일이 커서 변환에 시간이 더 걸린다

### 2.5 Drive 폴더 구조

```
MyDrive/2026/clone_voice/
├── recored_voice/                 # 참조 파일 + 대본을 두는 곳
│   ├── 애국가1절.m4a       # 음성 참조
│   ├── 애국가1절.txt
│   ├── 발표영상.mp4               # 영상만 넣어도 됨
│   ├── 발표영상.txt               # 없으면 STT가 만들어 줌
│   └── 애국가1절_16k.wav   # 자동 생성 (변환 캐시)
└── output/                        # 생성된 음성이 저장되는 곳
```

### 2.6 전체 흐름

```
이름 입력 ──▶ 이름.m4a / .mp4 ──ffmpeg──▶ 16kHz mono wav ──┬──────────────┐
                                                           │              │
                                    이름.txt 있음? ─ 예 ──▶ ref_text      │
                                          │                  ▲            ├──▶ generate_voice_clone ──▶ 결과 wav
                                          └─ 아니오 ─▶ Whisper STT ───────┤
                                                    (이름.txt 로 저장)    │
                                                                          │
생성할 문장 입력 ──────────────── target_text ────────────────────────────┘
```

참조 음성은 **한 번만** 불러오면 되고, 문장만 바꿔 가며 7번 셀을 반복 실행하면 된다.

## 3. Google Drive 연결

참조 음성과 대본을 Drive에서 읽어오므로 먼저 마운트한다.
실행하면 인증 창이 뜨고, 계정을 선택해 권한을 허용하면 된다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 4. 환경 설치

- `qwen-tts` — 모델 로드와 합성을 담당하는 공식 패키지
- `soundfile` — wav 읽기/쓰기
- `ffmpeg` — m4a·mp4 같은 원본을 16kHz 모노 wav로 변환

대본 자동 생성에 쓰는 Whisper는 `qwen-tts`가 함께 설치하는 `transformers`로
동작하므로 따로 설치할 패키지가 없다. 모델 가중치는 처음 쓸 때 내려받는다.

설치 후 런타임 재시작을 요구하면 재시작하고, **3번 셀부터** 다시 실행한다.

In [ ]:
!pip install -U qwen-tts soundfile
!apt-get -y install ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 569.8 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of gradio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.3/32.3 MB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 5. 모델 로드

Base 모델은 참조 음성을 따라 하는 용도의 체크포인트다.
GPU가 잡히면 `bfloat16`, 아니면 `float32`로 자동 설정된다.

첫 실행에서는 가중치(약 2.5GB)를 내려받으므로 몇 분 걸린다.
품질을 더 올리려면 `MODEL_ID`를 `Qwen/Qwen3-TTS-12Hz-1.7B-Base`로 바꾼다.
대신 VRAM을 더 쓰므로 무료 티어에서는 0.6B가 안전하다.

**런타임 → 런타임 유형 변경 → GPU** 설정을 먼저 확인할 것.

In [ ]:
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32

print("DEVICE:", DEVICE)
print("CUDA available:", torch.cuda.is_available())

model = Qwen3TTSModel.from_pretrained(
    MODEL_ID,
    device_map=DEVICE,
    dtype=DTYPE,
)

print("모델 로드 완료:", MODEL_ID)


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 
DEVICE: cuda:0
CUDA available: True


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.83G [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

모델 로드 완료: Qwen/Qwen3-TTS-12Hz-0.6B-Base


## 6. 참조 음성 준비

### 6.1 설정과 헬퍼 함수

경로를 바꿔야 한다면 **이 셀의 `VOICE_DIR` / `OUTPUT_DIR`만** 수정하면 된다.

이 셀이 정의하는 것:

- `find_media()` — 이름에 맞는 참조 파일을 음성·영상 확장자 순으로 찾는다
- `list_voices()` — 참조 파일 목록과 각각의 대본 유무를 돌려준다
- `read_text()` — 한글 txt는 UTF-8과 CP949가 섞여 있어 순서대로 시도한다
- `has_audio()` — ffprobe로 오디오 트랙 유무를 확인한다 (영상 파일 대비)
- `to_wav()` — 16kHz 모노 wav로 변환한다. 영상이면 `-vn`으로 오디오만 뽑고,
  이미 변환된 파일이 원본보다 최신이면 건너뛴다
- `get_stt()` / `transcribe()` — Whisper를 **처음 필요할 때만** 로드해 받아쓴다.
  대본이 이미 있으면 모델을 아예 올리지 않으므로 VRAM을 쓰지 않는다
- `load_reference()` — 이름 하나로 참조 파일과 대본을 읽어 `(wav 경로, ref_text)`를
  돌려준다. 대본이 없으면 STT로 만들어 저장한다

In [ ]:
import subprocess
from pathlib import Path
from datetime import datetime

import soundfile as sf
from IPython.display import Audio, display

# ── 설정 ────────────────────────────────────────────────
VOICE_DIR = Path("/content/drive/MyDrive/2026/clone_voice/recored_voice")
OUTPUT_DIR = Path("/content/drive/MyDrive/2026/clone_voice/output")
LANGUAGE = "Korean"
# 참조로 쓸 수 있는 확장자 (앞에 있는 것부터 우선 탐색)
AUDIO_EXTS = [".m4a", ".mp3", ".wav", ".webm", ".ogg", ".flac"]
VIDEO_EXTS = [".mp4", ".mov", ".mkv", ".avi"]
MEDIA_EXTS = AUDIO_EXTS + VIDEO_EXTS
CACHE_SUFFIX = "_16k"          # 변환 캐시 파일 표시 (목록에서 제외됨)
MIN_DURATION = 3.0

# .txt가 없을 때 Whisper로 대본을 자동 생성할지 여부
AUTO_STT = True
STT_MODEL_ID = "openai/whisper-large-v3-turbo"
STT_LANGUAGE = "ko"
# ───────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_media(name):
    """이름에 해당하는 참조 파일(음성 또는 영상) 경로를 찾는다. 없으면 None."""
    for ext in MEDIA_EXTS:
        p = VOICE_DIR / f"{name}{ext}"
        if p.exists():
            return p
    return None


def list_voices():
    """참조 파일이 있는 이름 → 대본(.txt) 존재 여부를 돌려준다."""
    found = {}
    for p in sorted(VOICE_DIR.iterdir()):
        if p.suffix.lower() not in MEDIA_EXTS:
            continue
        if p.stem.endswith(CACHE_SUFFIX):      # 변환 캐시는 제외
            continue
        found.setdefault(p.stem, (VOICE_DIR / f"{p.stem}.txt").exists())
    return found


def read_text(path):
    """한글 txt 인코딩을 순서대로 시도한다."""
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return path.read_text(encoding=enc).strip()
        except UnicodeDecodeError:
            continue
    raise ValueError(f"'{path.name}' 인코딩을 인식하지 못했습니다. UTF-8로 저장해 주세요.")


_stt_pipe = None


def get_stt():
    """Whisper 파이프라인을 처음 필요할 때 한 번만 로드한다."""
    global _stt_pipe
    if _stt_pipe is None:
        from transformers import pipeline
        print(f"STT 모델 로드 중: {STT_MODEL_ID}  (최초 1회, 수 분 소요)")
        _stt_pipe = pipeline(
            "automatic-speech-recognition",
            model=STT_MODEL_ID,
            device=0 if torch.cuda.is_available() else -1,
            dtype=DTYPE,
        )
    return _stt_pipe


def transcribe(wav_path):
    """참조 음성을 받아써서 문자열로 돌려준다."""
    result = get_stt()(
        str(wav_path),
        chunk_length_s=30,
        generate_kwargs={"language": STT_LANGUAGE, "task": "transcribe"},
    )
    return result["text"].strip()


def has_audio(src):
    """파일에 오디오 트랙이 실제로 들어 있는지 확인한다."""
    r = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "a",
         "-show_entries", "stream=codec_type", "-of", "csv=p=0", str(src)],
        capture_output=True, text=True,
    )
    return "audio" in r.stdout


def to_wav(src):
    """16kHz 모노 wav로 변환. 캐시가 원본보다 최신이면 건너뛴다.

    영상 파일(mp4 등)이면 -vn 으로 영상 트랙을 버리고 오디오만 추출한다.
    """
    dst = src.with_name(f"{src.stem}_16k.wav")
    if dst.exists() and dst.stat().st_mtime >= src.stat().st_mtime:
        return dst

    if not has_audio(src):
        raise ValueError(
            f"'{src.name}'에 오디오 트랙이 없습니다. 소리가 들어 있는 파일인지 확인하세요."
        )

    if src.suffix.lower() in VIDEO_EXTS:
        print(f"영상에서 오디오 추출 중: {src.name}")

    subprocess.run(
        ["ffmpeg", "-y", "-i", str(src), "-vn", "-ac", "1", "-ar", "16000", str(dst)],
        check=True, capture_output=True,
    )
    return dst


def load_reference(name, force_stt=False):
    """이름 하나로 참조 파일과 대본을 함께 읽는다.

    대본(.txt)이 없으면 AUTO_STT 설정에 따라 Whisper로 받아쓰고 .txt로 저장한다.
    force_stt=True면 기존 .txt가 있어도 다시 받아쓴다.
    """
    name = name.strip()
    if not name:
        raise ValueError("참조 음성 이름을 입력해야 합니다.")

    # 확장자까지 붙여서 입력한 경우도 받아준다
    if Path(name).suffix.lower() in MEDIA_EXTS + [".txt"]:
        name = Path(name).stem

    media = find_media(name)
    if media is None:
        raise FileNotFoundError(
            f"'{name}' 참조 파일을 찾을 수 없습니다.\n"
            f"  찾은 위치: {VOICE_DIR}\n"
            f"  지원 확장자: {', '.join(MEDIA_EXTS)}\n"
            f"  사용 가능한 이름: {list(list_voices()) or '없음'}"
        )

    wav = to_wav(media)
    data, sr = sf.read(wav)
    duration = len(data) / sr

    txt = VOICE_DIR / f"{name}.txt"
    if txt.exists() and not force_stt:
        ref_text = read_text(txt)
        source = "대본 파일"
    elif AUTO_STT or force_stt:
        print(f"대본을 음성에서 자동 생성합니다 ({name})")
        ref_text = transcribe(wav)
        txt.write_text(ref_text + "\n", encoding="utf-8")
        source = f"STT 자동 생성 → {txt.name} 에 저장됨"
    else:
        raise FileNotFoundError(
            f"'{name}.txt'가 없습니다. 대본을 같은 이름으로 두거나 AUTO_STT를 켜세요."
        )

    if not ref_text:
        raise ValueError(f"'{txt.name}'이 비어 있습니다.")

    kind = "영상" if media.suffix.lower() in VIDEO_EXTS else "음성"
    print(f"참조 파일  : {media.name} [{kind}] ({duration:.2f}초)")
    print(f"참조 텍스트: {ref_text}")
    print(f"대본 출처  : {source}")
    if duration < MIN_DURATION:
        print(f"경고: {MIN_DURATION}초 이상을 권장합니다. 짧으면 클로닝 품질이 떨어집니다.")

    return wav, ref_text


print("헬퍼 함수 준비 완료")
print("참조 파일 폴더:", VOICE_DIR)
print("출력 폴더     :", OUTPUT_DIR)
print("자동 STT      :", "켜짐" if AUTO_STT else "꺼짐", f"({STT_MODEL_ID})")

헬퍼 함수 준비 완료
참조 파일 폴더: /content/drive/MyDrive/2026/clone_voice/recored_voice
출력 폴더     : /content/drive/MyDrive/2026/clone_voice/output
자동 STT      : 켜짐 (openai/whisper-large-v3-turbo)


### 6.2 참조 음성 선택

폴더에서 사용 가능한 이름을 먼저 보여주고, 그중 하나를 입력받는다.
**확장자는 빼고** 이름만 입력한다. 예: `김남이_애국가1절`

`.m4a`든 `.mp4`든 확장자는 자동으로 찾으므로 입력할 때 신경 쓰지 않아도 된다.

목록이 비어 있으면 6.1의 `VOICE_DIR` 경로부터 확인한다.
매번 입력하기 번거로우면 `VOICE_NAME`에 직접 값을 넣어도 된다.

In [ ]:
voices = list_voices()
print("사용 가능한 참조 파일")
if voices:
    for n, has_txt in voices.items():
        print(f"  - {n}  [{'대본 있음' if has_txt else '대본 없음 → STT 자동 생성'}]")
else:
    print("  (없음 — VOICE_DIR 경로를 확인하세요)")

#VOICE_NAME = input("\n사용할 참조 음성 이름 (확장자 없이): ").strip()
VOICE_NAME = "라희_발표"   # 고정해서 쓰려면 이 줄을 사용

REF_WAV, REF_TEXT = load_reference(VOICE_NAME)

print("\n참조 음성 미리듣기")
display(Audio(str(REF_WAV)))

사용 가능한 참조 파일
  - 김남이_애국가1절  [대본 있음]
  - 김남이_통화_260914  [대본 있음]
  - 라희_발표  [대본 있음]
  - 마누라_통화_260914  [대본 있음]
참조 파일  : 라희_발표.mp4 [영상] (56.59초)
참조 텍스트: 안녕하세요. 3학년 3반 친구들. 이번학기 부반장 후보로 나온 김라희 입니다. 저는 우리 반 친구들이 매일매일 학교에 오고 싶어하는 즐겁고 편안한 반으로 만들기 위해서 다음과 같은 세가지 공약을 준비했습니다. 첫째, 23명의 친구들이 듣고 싶어하는 노래를 직접 정하여 그 노래를 점심시간에 직접 틀어주겠습니다. 둘째, 우리 반 친구들의 생일을 모두 조사하여 생일날 반 전체가 함께 생일 축하 노래는 불러주겠습니다. 셋째, 가끔 자나, 가위, 테이퍼 등의 짐을 두고 와서 당황한 적이 있으시죠? 교실의 공양 방문을 비치하여 친구들의 불법함을 주기겠습니다. 친구들의 이야기를 잘 듣고 우리 반을 위해 누구보다 열심히 뛰는 멋진 불법장이 되도록 하겠습니다. 금방에게 꼭 품어주세요. 감사합니다.
대본 출처  : 대본 파일

참조 음성 미리듣기


## 7. 음성 생성

생성할 문장은 여기서 직접 입력받는다.
참조 음성은 6번에서 이미 불러왔으므로, **문장만 바꿔 가며 이 셀을 반복 실행**하면 된다.
모델을 다시 로드하거나 파일을 다시 변환하지 않는다.

결과는 `output` 폴더에 `이름_날짜시각.wav`로 저장되고, 아래에서 바로 재생된다.

In [ ]:
if "REF_WAV" not in globals():
    raise RuntimeError("먼저 6.2 셀에서 참조 음성을 불러오세요.")

target_text = input("생성할 문장을 입력하세요:\n> ").strip()
if not target_text:
    raise ValueError("생성할 문장을 입력해야 합니다.")

print("\n생성 중...")
wavs, out_sr = model.generate_voice_clone(
    text=target_text,
    language=LANGUAGE,
    ref_audio=str(REF_WAV),
    ref_text=REF_TEXT,
)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = OUTPUT_DIR / f"{VOICE_NAME}_{stamp}.wav"
sf.write(out_path, wavs[0], out_sr)

print("생성 완료:", out_path)
display(Audio(str(out_path)))
# 내 이름은 OOO입니다. 만나서 반갑습니다. 우리 모두 좋은 세상에서 행복하게 살아요.

생성할 문장을 입력하세요:
> 저는 김라희라고 합니다. 만나서 반갑습니다. 우리 모두 좋은 세상에서 행복하게 살아요


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.



생성 중...
생성 완료: /content/drive/MyDrive/2026/clone_voice/output/라희_발표_20260915_053704.wav


## 8. 자주 막히는 지점

| 증상 | 원인과 조치 |
|---|---|
| 사용 가능한 목록이 비어 있음 | `VOICE_DIR` 경로 오타, 또는 `.m4a`와 `.txt` 이름이 서로 다름 |
| `FileNotFoundError` | 확장자를 뺀 이름만 입력했는지 확인. 한글 파일명은 띄어쓰기·언더바까지 정확히 일치해야 함 |
| 대본이 깨져서 출력됨 | txt를 UTF-8로 다시 저장 |
| `ffmpeg` 오류 | 4번 셀의 `apt-get install ffmpeg`가 실행됐는지 확인 |
| `오디오 트랙이 없습니다` | 무음 영상이거나 오디오가 분리된 파일. 소리가 나는지 먼저 재생해 볼 것 |
| mp4 변환이 오래 걸림 | 영상 파일이 크기 때문. 필요한 구간만 잘라서 올리면 빨라짐 |
| STT 결과가 부정확함 | 저장된 `이름.txt`를 직접 수정 후 6.2 재실행. 또는 `STT_MODEL_ID`를 큰 모델로 |
| STT가 엉뚱한 언어로 나옴 | `STT_LANGUAGE`를 확인 (한국어는 `ko`) |
| STT 로드 중 CUDA OOM | 대본을 직접 만들어 두거나 `STT_MODEL_ID`를 `openai/whisper-small`로 |
| 결과가 원본 목소리와 안 닮음 | `ref_text`가 녹음 내용과 정확히 일치하는지 확인. 3초 미만이거나 잡음이 많으면 품질이 떨어짐 |
| CUDA out of memory | 런타임 재시작 후 0.6B 모델 사용 |
| 생성이 매우 느림 | GPU 런타임이 아님. 런타임 유형을 GPU로 변경 |

### 참조 음성을 잘 만드는 요령

- 3초 이상, 10~20초 내외가 무난하다
- 배경 소음이 적고 한 사람만 말하는 구간을 쓴다
- 대본은 들리는 그대로 받아쓴다. 문장부호는 크게 중요하지 않다
- 영상을 쓸 때는 카메라 마이크 특성상 잔향이 섞이기 쉽다.
  가능하면 마이크 가까이서 녹음한 음성 파일이 결과가 낫다

### 주의

타인의 목소리를 복제할 때는 반드시 본인 동의를 받아야 한다.
동의 없는 음성 복제는 음성권 침해나 사기에 해당할 수 있다.